 # CS480: Introduction to Quantum Computing

 ## Homework 4: Multi-Qubit Circuits

 **Due:** Sunday 11:59 PM

 **Total Points:** 90



 ### Instructions

 - Complete all exercises in this notebook

 - Run all cells to verify your answers

 - Show your work and explain your reasoning where indicated

 - Submit the completed .ipynb file



 ### This homework covers:

 - Part 1: Tensor Products (20 pts)

 - Part 2: Multi-Qubit States in Qiskit (15 pts)

 - Part 3: CNOT Gate Exploration (20 pts)

 - Part 4: Circuit Construction (20 pts)

 - Part 5: Challenge Problems (15 pts)

In [51]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

sim = AerSimulator()


 ---

 ## Part 1: Tensor Products (20 points)

 ### Exercise 1.1 (5 points)

 Calculate the tensor product $|+\rangle \otimes |-\rangle$ using NumPy.

 Express your answer in the computational basis.



 Recall:

 - $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$

 - $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$

In [52]:
# TODO: Define |+⟩ and |−⟩ as numpy arrays
ket_plus = np.array([1,  1]) / np.sqrt(2)
ket_minus = np.array([1, -1]) / np.sqrt(2)

# TODO: Calculate |+⟩ ⊗ |−⟩ using np.kron
plus_minus = np.kron(ket_plus, ket_minus)

print("Exercise 1.1: |+⟩ ⊗ |−⟩")
print("Result:", plus_minus)

# YOUR ANSWER: Express |+−⟩ in computational basis form:
# |+−⟩ = _1/2__|00⟩  _-1/2__|01⟩ + _1/2__|10⟩  -_1/2__|11⟩


Exercise 1.1: |+⟩ ⊗ |−⟩
Result: [ 0.5 -0.5  0.5 -0.5]


 ### Exercise 1.2 (5 points)

 Calculate the tensor product $H \otimes X$ (Hadamard on first qubit, X on second)

 as a 4×4 matrix.

In [53]:
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]])
X = np.array([[0, 1], [1, 0]])

# TODO: Calculate H ⊗ X using np.kron
H_tensor_X = np.kron(H, X)

print("Exercise 1.2: H ⊗ X matrix")
print(np.round(H_tensor_X, 4))


Exercise 1.2: H ⊗ X matrix
[[ 0.      0.7071  0.      0.7071]
 [ 0.7071  0.      0.7071  0.    ]
 [ 0.      0.7071 -0.     -0.7071]
 [ 0.7071  0.     -0.7071 -0.    ]]


 ### Exercise 1.3 (5 points)

 Apply the operator $(H \otimes X)$ to the state $|00\rangle$.



 **First calculate manually:**

 $(H \otimes X)|00\rangle = (H|0\rangle) \otimes (X|0\rangle) = ?$



 Then verify with code.

In [54]:
ket_00 = np.array([1, 0, 0, 0])

# TODO: Apply H_tensor_X to ket_00
result_1_3 = H_tensor_X @ ket_00

print("Exercise 1.3: (H ⊗ X)|00⟩")
print("Result:", np.round(result_1_3, 4))

# YOUR ANSWER: What state is this in bra-ket notation?
#  (H⊗X)|00> = |+> ⊗ |1> = (|01> + |11>) / sqrt(2)


Exercise 1.3: (H ⊗ X)|00⟩
Result: [0.     0.7071 0.     0.7071]


 ### Exercise 1.4 (5 points)

 Prove mathematically that the state

 $$\frac{1}{2}(|00\rangle - |01\rangle + |10\rangle - |11\rangle)$$

 is a **product state**. Find the single-qubit states it factors into.



 **Your proof:**



 Let $|a\rangle = \alpha|0\rangle + \beta|1\rangle$ and $|b\rangle = \gamma|0\rangle + \delta|1\rangle$



 Then $|a\rangle \otimes |b\rangle = \alpha\gamma|00\rangle + \alpha\delta|01\rangle + \beta\gamma|10\rangle + \beta\delta|11\rangle$



 Matching coefficients:

 - $\alpha\gamma = 1/2$

 - $\alpha\delta = -1/2$

 - $\beta\gamma = 1/2$

 - $\beta\delta = -1/2$



 **TODO:** Solve for $\alpha, \beta, \gamma, \delta$ and identify $|a\rangle$ and $|b\rangle$.

In [55]:
# Verify your answer with code:
target_state = np.array([1, -1, 1, -1]) / 2

# TODO: Define the two single-qubit states you found
# ket_a = ?
# ket_b = ?
ket_a = np.array([1,  1]) / np.sqrt(2)  # |+>
ket_b = np.array([1, -1]) / np.sqrt(2)  # |->

# TODO: Compute tensor product and compare
# your_state = np.kron(ket_a, ket_b)
# print("Target:", target_state)
# print("Your tensor product:", your_state)
# print("Match:", np.allclose(target_state, your_state))
your_state = np.kron(ket_a, ket_b)
print('Target:', target_state)
print('Your tensor product:', your_state)
print('Match:', np.allclose(target_state, your_state))


Target: [ 0.5 -0.5  0.5 -0.5]
Your tensor product: [ 0.5 -0.5  0.5 -0.5]
Match: True


 ---

 ## Part 2: Multi-Qubit States in Qiskit (15 points)

 ### Exercise 2.1 (5 points)

 Create the following two-qubit states involving the $|-\rangle$ state:



 - a) $|-0\rangle = |-\rangle \otimes |0\rangle$ (q1 in $|-\rangle$, q0 in $|0\rangle$)

 - b) $|0-\rangle = |0\rangle \otimes |-\rangle$ (q1 in $|0\rangle$, q0 in $|-\rangle$)

 - c) $|--\rangle = |-\rangle \otimes |-\rangle$ (both qubits in $|-\rangle$)



 **Hint:** $|-\rangle = H|1\rangle$, so apply X then H.

In [56]:
print("Exercise 2.1: States with |−⟩")

# a) |−0⟩: |−⟩ on q1, |0⟩ on q0
qc_2_1a = QuantumCircuit(2)
# TODO: Create |−⟩ on q1
qc_2_1a.x(1)   # |1> on q1
qc_2_1a.h(1)   # |-> on q1

state_a = Statevector(qc_2_1a)
print("a) |−0⟩:", np.round(state_a.data, 4))
print("   Expected: [0.707, 0, -0.707, 0]")

# b) |0−⟩: |0⟩ on q1, |−⟩ on q0
qc_2_1b = QuantumCircuit(2)
# TODO: Create |−⟩ on q0
qc_2_1b.x(0)   # |1> on q0
qc_2_1b.h(0)   # |-> on q0

state_b = Statevector(qc_2_1b)
print("b) |0−⟩:", np.round(state_b.data, 4))
print("   Expected: [0.707, -0.707, 0, 0]")

# c) |−−⟩: |−⟩ on both qubits
qc_2_1c = QuantumCircuit(2)
# TODO: Create |−⟩ on both qubits
qc_2_1c.x(0)   # |1> on q0
qc_2_1c.h(0)   # |-> on q0
qc_2_1c.x(1)   # |1> on q1
qc_2_1c.h(1)   # |-> on q1

state_c = Statevector(qc_2_1c)
print("c) |−−⟩:", np.round(state_c.data, 4))
print("   Expected: [0.5, -0.5, -0.5, 0.5]")


Exercise 2.1: States with |−⟩
a) |−0⟩: [ 0.7071+0.j  0.    +0.j -0.7071+0.j -0.    +0.j]
   Expected: [0.707, 0, -0.707, 0]
b) |0−⟩: [ 0.7071+0.j -0.7071+0.j  0.    +0.j -0.    +0.j]
   Expected: [0.707, -0.707, 0, 0]
c) |−−⟩: [ 0.5+0.j -0.5+0.j -0.5+0.j  0.5+0.j]
   Expected: [0.5, -0.5, -0.5, 0.5]


 ### Exercise 2.2 (5 points)

 Create a state that is an **equal superposition of only three basis states**:

 $$\frac{1}{\sqrt{3}}(|00\rangle + |01\rangle + |10\rangle)$$



 This excludes $|11\rangle$ and **cannot** be created with just H gates!



 **Approach:** Use the `initialize()` method to directly set the statevector.

In [ ]:
qc_2_2 = QuantumCircuit(2)

target_amplitudes = [1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3), 0]
# TODO: Use qc_2_2.initialize(target_amplitudes) to set the state
qc_2_2.initialize(target_amplitudes)

print("Exercise 2.2: (|00⟩ + |01⟩ + |10⟩)/√3")
state_2_2 = Statevector(qc_2_2)
print("State:", np.round(state_2_2.data, 4))

# Add measurement and run
qc_2_2_measure = qc_2_2.copy()
qc_2_2_measure.measure_all()
result = sim.run(qc_2_2_measure, shots=3000).result()
counts = result.get_counts()
print("Counts (expect ~1000 each for 00, 01, 10; ~0 for 11):", counts)


 ### Exercise 2.3 (5 points)

 Create a **4-qubit cluster state** using CZ gates:



 ```python

 qc.h([0, 1, 2, 3])  # All qubits in |+⟩

 qc.cz(0, 1)

 qc.cz(1, 2)

 qc.cz(2, 3)

 ```



 - a) Build the circuit and print the statevector

 - b) Measure all qubits 1000 times. How many distinct outcomes appear?

 - c) Is this state entangled or a product state? Explain.

In [ ]:
print("Exercise 2.3: 4-qubit cluster state")

qc_2_3 = QuantumCircuit(4)
# TODO: Build the cluster state circuit
qc_2_3.h(0)
qc_2_3.h(1)
qc_2_3.h(2)
qc_2_3.h(3)
qc_2_3.cz(0, 1)
qc_2_3.cz(1, 2)
qc_2_3.cz(2, 3)

state_2_3 = Statevector(qc_2_3)
print("a) Statevector (first 8 amplitudes):", np.round(state_2_3.data[:8], 4))
print("   (remaining 8 amplitudes):", np.round(state_2_3.data[8:], 4))

# b) Measurement
qc_2_3_measure = qc_2_3.copy()
qc_2_3_measure.measure_all()
result = sim.run(qc_2_3_measure, shots=1000).result()
counts = result.get_counts()
print(f"b) Number of distinct outcomes: {len(counts)}")
print(f"   Counts: {counts}")


 **c) Is this state entangled or a product state? Explain:**

 YOUR ANSWER: This state is entangled, since all 16 basis states appear in the statevector with equal amplitude, wich 
means the qubits are correlated.


 ---

 ## Part 3: CNOT Gate Exploration (20 points)

 ### Exercise 3.1 (5 points)

 Verify the CNOT truth table by testing all four basis states.

 Use **q1 as control** and **q0 as target** (opposite of typical convention).

In [ ]:
print("Exercise 3.1: CNOT Truth Table (control=q1, target=q0)")
print("Input → Output")

for input_state in ['00', '01', '10', '11']:
    qc = QuantumCircuit(2)
    
    # TODO: Prepare the input state based on input_state string
    # Remember: input_state[0] is q1, input_state[1] is q0
    if input_state[1] == '1': qc.x(0)  # set q0
    if input_state[0] == '1': qc.x(1)  # set q1
    # TODO: Apply CNOT with control=q1, target=q0
    # qc.cx(1, 0)
    qc.cx(1, 0)
    state = Statevector(qc)
    # TODO: Determine the output state and print
    # Hint: use np.argmax(np.abs(state.data)) to find which basis state
    output_idx = np.argmax(np.abs(state.data))
    output_bits = format(output_idx, '02b')
    print(f'|{input_state}> -> |{output_bits}>')


 ### Exercise 3.2 (7 points)

 Explore what happens when we apply CNOT to **superposition states**.



 - a) What is $CNOT|+0\rangle$? (q0 in $|+\rangle$, q1 in $|0\rangle$, then CNOT with control=q0)

 - b) What is $CNOT|0+\rangle$? (q0 in $|0\rangle$, q1 in $|+\rangle$, then CNOT with control=q0)

 - c) Which one creates entanglement? Explain why.

In [ ]:
print("Exercise 3.2: CNOT on superposition states")

# a) |+0⟩ then CNOT(control=q0, target=q1)
qc_3_2a = QuantumCircuit(2)
# TODO: Create |+⟩ on q0, then apply CNOT
qc_3_2a.h(0)       # q0 in |+>
qc_3_2a.cx(0, 1)   # CNOT: control=q0, target=q1

print("a) CNOT|+0⟩ (control=q0 in |+⟩):")
print(qc_3_2a.draw('text'))
state_a = Statevector(qc_3_2a)
print("   State:", np.round(state_a.data, 4))

# b) |0+⟩ then CNOT(control=q0, target=q1)
qc_3_2b = QuantumCircuit(2)
# TODO: Create |+⟩ on q1, then apply CNOT
qc_3_2b.h(1)       # q1 in |+>
qc_3_2b.cx(0, 1)   # CNOT: control=q0, target=q1

print("b) CNOT|0+⟩ (control=q0 in |0⟩):")
print(qc_3_2b.draw('text'))
state_b = Statevector(qc_3_2b)
print("   State:", np.round(state_b.data, 4))


 **c) Which creates entanglement and why?**



 *Hint: Try to factor each result as a tensor product.*



YOUR ANSWER: a) CNOT|+0> creates entanglement. The result is (|00> + |11>)/sqrt(2) which is a Bell state.
You cannot factor this into two separate single-qubit states.
The CNOT flips q1 only when q0=|1>, so the two qubits become correlated.




 ### Exercise 3.3 (8 points)

 The **XOR interpretation** of CNOT: $CNOT|a,b\rangle = |a, a \oplus b\rangle$



 where $\oplus$ is XOR (addition mod 2).



 **a)** Using this formula, show step-by-step how CNOT creates a Bell state

 from $(|0\rangle+|1\rangle)/\sqrt{2} \otimes |0\rangle$.



 **b)** What state does CNOT produce from $(|0\rangle+|1\rangle)/\sqrt{2} \otimes |1\rangle$?

 Calculate using XOR interpretation and verify with code.

 **a) Your derivation:**



 Start: $(|0\rangle+|1\rangle)/\sqrt{2} \otimes |0\rangle = (|00\rangle + |10\rangle)/\sqrt{2}$



 Apply $CNOT|a,b\rangle = |a, a\oplus b\rangle$ to each term:

 - $CNOT|00\rangle = |0, 0\oplus 0\rangle = $ |00⟩

 - $CNOT|10\rangle = |1, 1\oplus 0\rangle = $ |11⟩



 Final state: (|00⟩ + |11⟩)/√2



 This is the Bell state because: This is the Bell state because it is an equal superposition of |00⟩ and |11⟩
and cannot be written as a product of two single-qubit states.

In [ ]:
# b) CNOT applied to |+1⟩ = (|0⟩+|1⟩)/√2 ⊗ |1⟩
print("Exercise 3.3b: CNOT|+1⟩")

# YOUR PREDICTION using XOR:
# Start: (|01⟩ + |11⟩)/√2
# Apply CNOT (control=q0, target=q1), so |q1,q0⟩ -> |q1⊕q0, q0⟩:
# - CNOT|01⟩ = |0⊕1, 1⟩ = |11⟩
# - CNOT|11⟩ = |1⊕1, 1⟩ = |01⟩
# Predicted final state: (|01⟩ + |10⟩)/√2


# Verify with code:
qc_3_3b = QuantumCircuit(2)
# TODO: Create |+1⟩ then apply CNOT
qc_3_3b.x(1)      # q1 in |1>
qc_3_3b.h(0)      # q0 in |+>
qc_3_3b.cx(0, 1)  # CNOT: control=q0, target=q1

state_3_3b = Statevector(qc_3_3b)
print("Result:", np.round(state_3_3b.data, 4))
print("Your prediction matched?")
print('Expected: (|01> + |10>)/sqrt(2) = [0, 0.707, 0.707, 0]')
print('Prediction matched?', np.allclose(np.abs(state_3_3b.data), [0, 1/np.sqrt(2), 1/np.sqrt(2), 0]))



 ---

 ## Part 4: Circuit Construction (20 points)

 ### Exercise 4.1 (7 points)

 Show that you can **reverse the direction of a CNOT** using Hadamard gates:



 $$(H \otimes H) \cdot CNOT_{0 \to 1} \cdot (H \otimes H) = CNOT_{1 \to 0}$$



 This is useful when hardware only supports CNOT in one direction!



 - a) Build both circuits and verify they produce the same Operator matrix

 - b) Test both on the state $|10\rangle$ to confirm the behavior

 - c) Explain why this identity works

In [ ]:
print("Exercise 4.1: CNOT direction reversal")

# Circuit 1: Direct CNOT(1→0), i.e., q1 controls q0
qc_direct = QuantumCircuit(2)
qc_direct.cx(1, 0)

# Circuit 2: H-H, then CNOT(0→1), then H-H
qc_reversed = QuantumCircuit(2)
# TODO: Apply H to both qubits
# TODO: Apply CNOT with q0 as control, q1 as target
# TODO: Apply H to both qubits again
qc_reversed.h(0)
qc_reversed.h(1)
qc_reversed.cx(0, 1)
qc_reversed.h(0)
qc_reversed.h(1)

print("a) Direct CNOT(1→0):")
print(qc_direct.draw('text'))
print("\n   Reversed using H gates:")
print(qc_reversed.draw('text'))

print("\n   Matrices equal?", Operator(qc_direct).equiv(Operator(qc_reversed)))


In [ ]:
# b) Test on |10⟩
print("b) Testing on |10⟩:")

qc_test1 = QuantumCircuit(2)
qc_test1.x(1)  # Create |10⟩
qc_test1.cx(1, 0)  # Direct CNOT(1→0)
print("   Direct CNOT(1→0)|10⟩ =", Statevector(qc_test1).data)

qc_test2 = QuantumCircuit(2)
qc_test2.x(1)  # Create |10⟩
# TODO: Apply the H-CNOT-H sequence
qc_test2.h(0)
qc_test2.h(1)
qc_test2.cx(0, 1)
qc_test2.h(0)
qc_test2.h(1)
print("   H-CNOT-H |10⟩ =", Statevector(qc_test2).data)


 **c) Why does $(H \otimes H) \cdot CNOT \cdot (H \otimes H)$ reverse the control and target?**



 *Hint: Think about what H does to $|0\rangle$ and $|1\rangle$, and how CNOT acts in the X-basis.*



 YOUR ANSWER: H converts between the Z-basis (|0>,|1>) and the X-basis (|+>,|->).
In the X-basis, the roles of control and target are swapped for CNOT.
So applying H before and after effectively makes q1 the control and q0 the target,
reversing the direction of the CNOT.



 ### Exercise 4.2 (6 points)

 Create the state $\frac{1}{\sqrt{2}}(|010\rangle + |101\rangle)$



 **Hint:** Notice that these states are bitwise complements of each other.

 Think about how to use superposition and controlled operations.

In [ ]:
qc_4_2 = QuantumCircuit(3)
# TODO: Create (|010⟩ + |101⟩)/√2
qc_4_2.h(0)
qc_4_2.cx(0, 1)
qc_4_2.cx(0, 2)
qc_4_2.x(1)

print("Exercise 4.2: Create (|010⟩ + |101⟩)/√2")
print(qc_4_2.draw('text'))
state_4_2 = Statevector(qc_4_2)
print("Statevector:", np.round(state_4_2.data, 4))
print("Expected: non-zero at indices 2 (010) and 5 (101) with amplitude ≈0.707")

# Verify with measurement
qc_4_2_measure = qc_4_2.copy()
qc_4_2_measure.measure_all()
result = sim.run(qc_4_2_measure, shots=1000).result()
print("Measurement counts:", result.get_counts())


 ### Exercise 4.3 (7 points)

 Implement the **Fredkin gate** (controlled-SWAP):

 Swap q1 and q2 **only if** q0 = $|1\rangle$.



 | Input | Output |

 |-------|--------|

 | $\|000\rangle$ | $\|000\rangle$ |

 | $\|001\rangle$ | $\|001\rangle$ |

 | $\|010\rangle$ | $\|010\rangle$ |

 | $\|011\rangle$ | $\|011\rangle$ |

 | $\|100\rangle$ | $\|100\rangle$ |

 | $\|101\rangle$ | $\|110\rangle$ ← swapped! |

 | $\|110\rangle$ | $\|101\rangle$ ← swapped! |

 | $\|111\rangle$ | $\|111\rangle$ |



 - a) Implement using `qc.cswap(control, target1, target2)`

 - b) Verify the truth table for all 8 basis states

In [ ]:
print("Exercise 4.3: Fredkin gate (controlled-SWAP)")

# a) Build the Fredkin gate circuit
qc_fredkin = QuantumCircuit(3)
# TODO: Use qc_fredkin.cswap(0, 1, 2)
qc_fredkin.cswap(0, 1, 2)  # control=q0, swap q1 and q2

# b) Verify truth table
print("Truth table verification:")
print("Input → Output")

for i in range(8):
    input_bits = format(i, '03b')
    
    qc_test = QuantumCircuit(3)
    # Prepare input state
    if input_bits[2] == '1': qc_test.x(0)  # q0
    if input_bits[1] == '1': qc_test.x(1)  # q1
    if input_bits[0] == '1': qc_test.x(2)  # q2
    
    # TODO: Apply Fredkin gate
    qc_test.cswap(0, 1, 2)  # Apply Fredkin gate
    
    state = Statevector(qc_test)
    output_idx = np.argmax(np.abs(state.data))
    output_bits = format(output_idx, '03b')
    
    swapped = " ← swapped!" if input_bits in ['101', '110'] else ""
    print(f"|{input_bits}⟩ → |{output_bits}⟩{swapped}")


 ---

 ## Part 5: Challenge Problems (15 points)

 ### Challenge 5.1 (7 points)

 Prove that **CZ is symmetric** (doesn't matter which qubit is "control").



 - a) Show that `CZ(0,1)` and `CZ(1,0)` produce the same matrix

 - b) Explain WHY CZ is symmetric but CNOT is not

In [ ]:
print("Challenge 5.1: CZ symmetry")

# a) Compare CZ both ways
qc_cz_01 = QuantumCircuit(2)
qc_cz_01.cz(0, 1)

qc_cz_10 = QuantumCircuit(2)
qc_cz_10.cz(1, 0)

print("a) CZ(0,1) matrix:")
print(Operator(qc_cz_01).data.astype(int))
print("\n   CZ(1,0) matrix:")
print(Operator(qc_cz_10).data.astype(int))
print("\n   Are they equal?", Operator(qc_cz_01).equiv(Operator(qc_cz_10)))


 **b) Why is CZ symmetric but CNOT is not?**



 *Hint: Think about what each gate does to $|11\rangle$. What's special about phase vs bit flip?*



 YOUR ANSWER: CZ only does something when both qubits are |1⟩, it adds a -1 phase to that term.
Since both qubits have to be |1⟩ anyway, it doesn't matter which one you call control.

CNOT flips one specific qubit (the target) based on the other (the control).
Flipping q0 vs flipping q1 gives different results, so the order matters.



 ### Challenge 5.2 (8 points)

 Implement **CZ using only CNOT and single-qubit gates**.



 The identity is: $CZ = (I \otimes H) \cdot CNOT \cdot (I \otimes H)$



 - a) Build the circuit

 - b) Verify the matrices match

 - c) Test on $|++\rangle$ state and identify the resulting state

In [ ]:
print("Challenge 5.2: CZ from CNOT")

# Built-in CZ for reference
qc_cz_native = QuantumCircuit(2)
qc_cz_native.cz(0, 1)

# Your implementation using H and CNOT
qc_cz_built = QuantumCircuit(2)
# TODO: Build CZ using H on target, CNOT, H on target
qc_cz_built.h(1)      # H on target qubit
qc_cz_built.cx(0, 1)  # CNOT
qc_cz_built.h(1)      # H on target qubit again

print("a) Your CZ implementation:")
print(qc_cz_built.draw('text'))

print("\nb) Matrices match?", Operator(qc_cz_built).equiv(Operator(qc_cz_native)))


In [ ]:
# c) Test on |++⟩
print("c) Apply CZ to |++⟩:")

qc_test_native = QuantumCircuit(2)
qc_test_native.h(0)
qc_test_native.h(1)
qc_test_native.cz(0, 1)
print("   Native CZ|++⟩:", np.round(Statevector(qc_test_native).data, 4))

qc_test_built = QuantumCircuit(2)
qc_test_built.h(0)
qc_test_built.h(1)
# TODO: Apply your CZ implementation
qc_test_built.h(1)      # H on target
qc_test_built.cx(0, 1)  # CNOT
qc_test_built.h(1)      # H on target
print("   Your CZ|++⟩:", np.round(Statevector(qc_test_built).data, 4))


 **What state is $CZ|++\rangle$? Can you identify it?**



 *Hint: Compare with the Bell states.*



 YOUR ANSWER: This is not a Bell state. The -1 on |11⟩ comes from CZ flipping the phase
of that term. All four basis states appear, so it cannot be factored into
two separate qubits, it is entangled.



 ---

 ## Submission Checklist



 Before submitting, verify:



 - [ ] All code cells run without errors

 - [ ] All TODO sections are completed

 - [ ] Written explanations provided where requested

 - [ ] File saved as .ipynb format



 ### Grading

 | Part | Topic | Points |

 |------|-------|--------|

 | 1 | Tensor Products | 20 |

 | 2 | Multi-Qubit States in Qiskit | 15 |

 | 3 | CNOT Gate Exploration | 20 |

 | 4 | Circuit Construction | 20 |

 | 5 | Challenge Problems | 15 |

 | **Total** | | **90** |